In [ ]:
# Author: mostly Rick den Otter, adapted by me. 
# preprocessing

In [1]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import os
import mne
import re
from mne_bids import read_raw_bids, BIDSPath
import pandas as pd
from gedai import Gedai

In [2]:
#Load data path

##DATA_PATH = Path(os.getenv("DATA_PATH"))

DATA_PATH = Path(r"C:\Scriptie_Data") 
path = DATA_PATH / "rps"

In [5]:
pars = pd.read_csv(path / "participants_current.tsv", sep="\t")
# Taken from biosemi64.lay (fieldtrip layout file, found that the authors also use it here: https://osf.io/yjxkn/files/5dq83)
ch_names = [
    'Fp1', 'AF7', 'AF3', 'F1', 'F3', 'F5', 'F7', 'FT7',
    'FC5', 'FC3', 'FC1', 'C1', 'C3', 'C5', 'T7', 'TP7',
    'CP5', 'CP3', 'CP1', 'P1', 'P3', 'P5', 'P7', 'P9',
    'PO7', 'PO3', 'O1', 'Iz', 'Oz', 'POz', 'Pz', 'CPz',
    'Fpz', 'Fp2', 'AF8', 'AF4', 'AFz', 'Fz', 'F2', 'F4',
    'F6', 'F8', 'FT8', 'FC6', 'FC4', 'FC2', 'FCz', 'Cz',
    'C2', 'C4', 'C6', 'T8', 'TP8', 'CP6', 'CP4', 'CP2',
    'P2', 'P4', 'P6', 'P8', 'P10', 'PO8', 'PO4', 'O2'
]

for row in pars.itertuples():
    p_id = row.participant_id
    subject = p_id.split("-")[1]
    bids_path = BIDSPath(subject=subject, task="RPS", suffix="eeg", datatype='eeg', root=path)
    if bids_path.fpath.exists():
        print(f"Loading {bids_path.fpath}")

        metadata_path = bids_path.copy().update(suffix='events', extension='.tsv')
        metadata = pd.read_csv(metadata_path.fpath, sep="\t")
        metadata.drop(columns=["onset", "duration", "onset_sample"], inplace=True)
        
        # Puts events from .tsv files into the raw object as annotations
        raw = read_raw_bids(bids_path, extra_params={"preload": True}, verbose=False)
        
        raw.annotations.onset += 2 # Add 2 seconds to make events centred on response event
        raw.annotations.rename({"n/a": "decision"})

        # Resample to 200 Hz
        raw.resample(200)

        # Bandpass filter between 0.1 and 40 Hz
        raw.filter(0.1, 40)

        events = mne.events_from_annotations(raw)

        # Split into p1 and p2 raws, fix montage, handle bads according to original author
        player1_raw = raw.copy().pick([ch for ch in raw.ch_names if re.match(r"^1-[AB]", ch)])
        rename_dict = dict(zip(player1_raw.ch_names, ch_names))
        player1_raw.rename_channels(rename_dict)
        player1_raw.set_montage("biosemi64")
        bad_chs_p1 = [ch.strip() for ch in row.player1_pre_processing_channels_fixed.split(",")] if pd.notna(row.player1_pre_processing_channels_fixed) else []
        player1_raw.info['bads'] = bad_chs_p1
        player1_raw.set_eeg_reference("average", projection=False)
        player1_raw.interpolate_bads()

        player2_raw = raw.pick([ch for ch in raw.ch_names if re.match(r"^2-[AB]", ch)])
        rename_dict = dict(zip(player2_raw.ch_names, ch_names))
        player2_raw.rename_channels(rename_dict)
        player2_raw.set_montage("biosemi64")
        bad_chs_p2 = [ch.strip() for ch in row.player2_pre_processing_channels_fixed.split(",")] if pd.notna(row.player2_pre_processing_channels_fixed) else []
        player2_raw.info['bads'] = bad_chs_p2
        player2_raw.set_eeg_reference("average", projection=False)
        player2_raw.interpolate_bads()

        # Apply GEDAI (https://www.biorxiv.org/content/10.1101/2025.10.04.680449v1) for easy automatic artifact repair
        print("Applying GEDAI - Player 1...")
        ged = Gedai()
        ged.fit_raw(player1_raw, overlap=0.0)
        player1_raw = ged.transform_raw(player1_raw)

        print("Applying GEDAI - Player 2...")
        ged = Gedai()
        ged.fit_raw(player2_raw, overlap=0.0)
        player2_raw = ged.transform_raw(player2_raw)

        # Epoch from [-0.2, 2] around presentation of the response stimulus, since response time is capped to 2 seconds.
        # Baseline using [-0.2, 0]
        player1_epochs = mne.Epochs(player1_raw, events[0], event_id=events[1], tmin=-0.2, tmax=2.0, baseline=(-0.2, 0), metadata=metadata)

        player2_epochs = mne.Epochs(player2_raw, events[0], event_id=events[1], tmin=-0.2, tmax=2.0, baseline=(-0.2, 0), metadata=metadata)

        player1_epochs.save(path / f"sub-{subject}_player-1_epo.fif", overwrite=True)
        player2_epochs.save(path / f"sub-{subject}_player-2_epo.fif", overwrite=True)
    else:
        print(f"File {bids_path.fpath} does not exist.")


Loading C:\Scriptie_Data\rps\sub-09\eeg\sub-09_task-RPS_eeg.bdf
Finding events on: Status


C:\Users\mathi\AppData\Local\Temp\ipykernel_19924\3843049618.py:26: RuntimeWarning: Did not find any channels.tsv associated with sub-09_task-RPS.

The search_str was "C:\Scriptie_Data\rps\sub-09\**\eeg\sub-09*channels.tsv"
  raw = read_raw_bids(bids_path, extra_params={"preload": True}, verbose=False)
C:\Users\mathi\AppData\Local\Temp\ipykernel_19924\3843049618.py:26: RuntimeWarning: Unable to map the following column(s) to to MNE:
date: 14/12/2021
player1_age: 25
player1_gender: F
player1_handedness: R
player1_pre_processing_channels_fixed: 
player2_age: 34
player2_gender: F
player2_handedness: R
player2_pre_processing_channels_fixed: "P1, O2"
  raw = read_raw_bids(bids_path, extra_params={"preload": True}, verbose=False)


Trigger channel Status has a non-zero initial value of 80639 (consider using initial_event=True to detect this event)
481 events found on stim channel Status
Event IDs: [  511 15103]
Finding events on: Status
Trigger channel Status has a non-zero initial value of 80639 (consider using initial_event=True to detect this event)
481 events found on stim channel Status
Event IDs: [  511 15103]
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.10
- Lower transition bandwidth: 0.10 Hz (-6 dB cutoff frequency: 0.05 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 6601 samples (33.005 s)

Used Annotations descriptions: ['decision']

C:\Users\mathi\AppData\Local\Temp\ipykernel_19924\3843049618.py:47: RuntimeWarning: No bad channels to interpolate. Doing nothing...
  player1_raw.interpolate_bads()


Setting channel interpolation method to {'eeg': 'spline'}.
Interpolating bad channels.
    Automatic origin fit: head of radius 95.0 mm
Computing interpolation matrix from 62 sensor positions
Interpolating 2 sensors
Applying GEDAI - Player 1...
Applying GEDAI - Player 2...
Adding metadata with 6 columns
480 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Adding metadata with 6 columns
480 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 480 events and 441 original time points ...
0 bad epochs dropped
Using data from preloaded Raw for 1 events and 441 original time points ...
Using data from preloaded Raw for 480 events and 441 original time points ...
Using data from preloaded Raw for 480 events and 441 original time points ...
0 bad epochs dropped
Using data from preloaded Raw for 1 events and 441 original time points ...
Using data from preloaded Raw for 480 even

...

Finding events on: Status
Trigger channel Status has a non-zero initial value of 65791 (consider using initial_event=True to detect this event)
480 events found on stim channel Status
Event IDs: [511]


C:\Users\mathi\AppData\Local\Temp\ipykernel_19924\3843049618.py:26: RuntimeWarning: Unable to map the following column(s) to to MNE:
date: 31/3/2023
player1_age: 32
player1_gender: F
player1_handedness: R
player1_pre_processing_channels_fixed: TP7
player2_age: 29
player2_gender: NB
player2_handedness: R
player2_pre_processing_channels_fixed: 
  raw = read_raw_bids(bids_path, extra_params={"preload": True}, verbose=False)


Finding events on: Status
Trigger channel Status has a non-zero initial value of 65791 (consider using initial_event=True to detect this event)
480 events found on stim channel Status
Event IDs: [511]
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.10
- Lower transition bandwidth: 0.10 Hz (-6 dB cutoff frequency: 0.05 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 6601 samples (33.005 s)

Used Annotations descriptions: ['decision']
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Setting channel interpolation method to {'eeg': 'spline'}.
Interpolating bad ch

C:\Users\mathi\AppData\Local\Temp\ipykernel_19924\3843049618.py:56: RuntimeWarning: No bad channels to interpolate. Doing nothing...
  player2_raw.interpolate_bads()


Applying GEDAI - Player 2...
Adding metadata with 6 columns
480 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Adding metadata with 6 columns
480 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 480 events and 441 original time points ...
0 bad epochs dropped
Using data from preloaded Raw for 1 events and 441 original time points ...
Using data from preloaded Raw for 480 events and 441 original time points ...
Using data from preloaded Raw for 480 events and 441 original time points ...
0 bad epochs dropped
Using data from preloaded Raw for 1 events and 441 original time points ...
Using data from preloaded Raw for 480 events and 441 original time points ...
